# Part2 원천 데이터 전처리

목표: Training_Original / Validation_Original의 Part2 ZIP 파일들을
압축 해제 → JSON 파싱 → CSV 저장

기존 라벨링 전처리 코드와 동일한 방식, Part2 원천 전용

## 1. Google Drive 연결

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


## 2. 경로 설정

In [2]:
import os

TR_DATA_PATH = '/content/drive/MyDrive/2026 4-1 캡스톤/DATA/Training_Original'
VL_DATA_PATH = '/content/drive/MyDrive/2026 4-1 캡스톤/DATA/Validation_Original'

# 경로 확인
for label, path in [('Training_Original', TR_DATA_PATH), ('Validation_Original', VL_DATA_PATH)]:
    if os.path.exists(path):
        files = os.listdir(path)
        part2_zips = [f for f in files if 'Part2' in f and f.endswith('.zip')]
        print(f'✅ {label}: 전체 {len(files)}개 파일, Part2 ZIP {len(part2_zips)}개')
    else:
        print(f'❌ {label} 경로 없음: {path}')

✅ Training_Original: 전체 23개 파일, Part2 ZIP 21개
✅ Validation_Original: 전체 22개 파일, Part2 ZIP 21개


## 3. ZIP 압축 해제

In [3]:
import zipfile
import glob

# 압축 해제 경로
TR_EXTRACT_PATH = '/content/tr_original_data'
VL_EXTRACT_PATH = '/content/vl_original_data'
os.makedirs(TR_EXTRACT_PATH, exist_ok=True)
os.makedirs(VL_EXTRACT_PATH, exist_ok=True)

def extract_part2_zips(data_path, extract_path):
    """Part2 ZIP 파일만 골라서 압축 해제"""
    zip_files = [
        f for f in glob.glob(os.path.join(data_path, '*.zip'))
        if 'Part2' in os.path.basename(f)
    ]
    print(f'  Part2 ZIP {len(zip_files)}개 발견')

    for i, zf in enumerate(zip_files):
        fname = os.path.basename(zf)
        print(f'  [{i+1}/{len(zip_files)}] {fname} 압축 해제 중...')
        try:
            with zipfile.ZipFile(zf, 'r') as z:
                z.extractall(extract_path)
        except Exception as e:
            print(f'    ⚠️ 오류: {e}')

    json_files = glob.glob(os.path.join(extract_path, '**', '*.json'), recursive=True)
    print(f'  ✅ 압축 해제 완료! JSON {len(json_files):,}개')
    return json_files

print('=== Training 압축 해제 ===')
tr_json_files = extract_part2_zips(TR_DATA_PATH, TR_EXTRACT_PATH)

print('\n=== Validation 압축 해제 ===')
vl_json_files = extract_part2_zips(VL_DATA_PATH, VL_EXTRACT_PATH)

=== Training 압축 해제 ===
  Part2 ZIP 21개 발견
  [1/21] TS_Part2_Clickbait_Auto_EC.zip 압축 해제 중...
  [2/21] TS_Part2_Clickbait_Auto_ET.zip 압축 해제 중...
  [3/21] TS_Part2_Clickbait_Auto_GB.zip 압축 해제 중...
  [4/21] TS_Part2_Clickbait_Auto_IS.zip 압축 해제 중...
  [5/21] TS_Part2_Clickbait_Auto_LC.zip 압축 해제 중...
  [6/21] TS_Part2_Clickbait_Auto_PO.zip 압축 해제 중...
  [7/21] TS_Part2_Clickbait_Auto_SO.zip 압축 해제 중...
  [8/21] TS_Part2_Clickbait_Direct_EC.zip 압축 해제 중...
  [9/21] TS_Part2_Clickbait_Direct_ET.zip 압축 해제 중...
  [10/21] TS_Part2_Clickbait_Direct_IS.zip 압축 해제 중...
  [11/21] TS_Part2_Clickbait_Direct_GB.zip 압축 해제 중...
  [12/21] TS_Part2_Clickbait_Direct_PO.zip 압축 해제 중...
  [13/21] TS_Part2_Clickbait_Direct_LC.zip 압축 해제 중...
  [14/21] TS_Part2_Clickbait_Direct_SO.zip 압축 해제 중...
  [15/21] TS_Part2_NonClickbait_Auto_EC.zip 압축 해제 중...
  [16/21] TS_Part2_NonClickbait_Auto_ET.zip 압축 해제 중...
  [17/21] TS_Part2_NonClickbait_Auto_GB.zip 압축 해제 중...
  [18/21] TS_Part2_NonClickbait_Auto_IS.zip 압축 해제 중...
  [19

## 4. JSON 파싱 함수 정의

In [4]:
import json

def parse_part2_original(filepath):
    """
    Part2 원천 데이터 JSON 파싱
    - sourceDataInfo에서 기본 정보 + processSentencenum + sentenceInfo 추출
    - labeledDataInfo가 없으므로 clickbaitClass는 useType으로 판단 (0=낚시성, 1=정상)
    """
    try:
        with open(filepath, 'r', encoding='utf-8') as f:
            data = json.load(f)

        source = data.get('sourceDataInfo', {})

        # Part2만 처리
        if source.get('partNum') != 'P2':
            return None

        # 기본 정보
        news_id          = source.get('newsID', '')
        sentence_count   = source.get('sentenceCount', 0)
        process_sent_num = source.get('processSentencenum', 0)  # 삽입 위치 (정상=0)
        sentences        = source.get('sentenceInfo', [])

        # clickbaitClass: useType 기준 (0=낚시성, 1=정상)
        # useType이 없으면 processSentencenum으로 판단 (0=정상, >0=낚시성)
        use_type = source.get('useType', None)
        if use_type is not None:
            clickbait_class = use_type  # 0=낚시성, 1=정상
        else:
            clickbait_class = 0 if process_sent_num > 0 else 1

        # 삽입 위치 비율 (정상이면 0.0)
        insert_position_ratio = (
            process_sent_num / sentence_count
            if sentence_count > 0 and process_sent_num > 0
            else 0.0
        )

        # 삽입 문장 / 앞뒤 문장 텍스트 추출
        inserted_sent = ''
        prev_sent     = ''
        next_sent     = ''

        if process_sent_num > 0 and sentences:
            # sentenceNo는 1-based
            sent_map = {s['sentenceNo']: s['sentenceContent'] for s in sentences}
            inserted_sent = sent_map.get(process_sent_num, '')
            prev_sent     = sent_map.get(process_sent_num - 1, '')  # 첫 문장이면 ''
            next_sent     = sent_map.get(process_sent_num + 1, '')  # 마지막 문장이면 ''

        return {
            'newsID'              : news_id,
            'newsCategory'        : source.get('newsCategory', ''),
            'newsSubcategory'     : source.get('newsSubcategory', ''),
            'newsTitle'           : source.get('newsTitle', ''),
            'newsContent'         : source.get('newsContent', ''),
            'partNum'             : 'P2',
            'processType'         : source.get('processType', ''),
            'processLevel'        : source.get('processLevel', ''),
            'sentenceCount'       : sentence_count,
            'clickbaitClass'      : clickbait_class,
            'processSentencenum'  : process_sent_num,
            'insertPositionRatio' : insert_position_ratio,
            'insertedSent'        : inserted_sent,
            'prevSent'            : prev_sent,
            'nextSent'            : next_sent,
        }

    except Exception as e:
        return None

print('✅ 파싱 함수 정의 완료')

✅ 파싱 함수 정의 완료


## 5. Training JSON 전체 파싱

In [5]:
import pandas as pd
from tqdm import tqdm

print('🔄 Training JSON 파싱 중...')
tr_rows = []
tr_errors = 0

for filepath in tqdm(tr_json_files, desc='TR 파싱'):
    result = parse_part2_original(filepath)
    if result:
        tr_rows.append(result)
    else:
        tr_errors += 1

tr_df = pd.DataFrame(tr_rows)
print(f'\n✅ TR 파싱 완료!')
print(f'  성공: {len(tr_df):,}건 / 실패: {tr_errors:,}건')
print(f'  컬럼: {list(tr_df.columns)}')

# 라벨 분포 확인
print(f'\n[TR 라벨 분포]')
print(tr_df['clickbaitClass'].value_counts().to_string())

# 핵심 확인: processSentencenum 분포
print(f'\n[TR processSentencenum] 낚시성 vs 정상')
print(tr_df.groupby('clickbaitClass')['processSentencenum'].describe())

🔄 Training JSON 파싱 중...


TR 파싱: 100%|██████████| 295275/295275 [00:55<00:00, 5329.16it/s]



✅ TR 파싱 완료!
  성공: 295,275건 / 실패: 0건
  컬럼: ['newsID', 'newsCategory', 'newsSubcategory', 'newsTitle', 'newsContent', 'partNum', 'processType', 'processLevel', 'sentenceCount', 'clickbaitClass', 'processSentencenum', 'insertPositionRatio', 'insertedSent', 'prevSent', 'nextSent']

[TR 라벨 분포]
clickbaitClass
0    149928
1    145347

[TR processSentencenum] 낚시성 vs 정상
                   count      mean       std  min  25%  50%  75%   max
clickbaitClass                                                        
0               149928.0  2.401806  1.397452  1.0  1.0  2.0  3.0  14.0
1               145347.0  0.000000  0.000000  0.0  0.0  0.0  0.0   0.0


## 6. Validation JSON 전체 파싱

In [6]:
print('🔄 Validation JSON 파싱 중...')
vl_rows = []
vl_errors = 0

for filepath in tqdm(vl_json_files, desc='VL 파싱'):
    result = parse_part2_original(filepath)
    if result:
        vl_rows.append(result)
    else:
        vl_errors += 1

vl_df = pd.DataFrame(vl_rows)
print(f'\n✅ VL 파싱 완료!')
print(f'  성공: {len(vl_df):,}건 / 실패: {vl_errors:,}건')

# 라벨 분포 확인
print(f'\n[VL 라벨 분포]')
print(vl_df['clickbaitClass'].value_counts().to_string())

print(f'\n[VL processSentencenum] 낚시성 vs 정상')
print(vl_df.groupby('clickbaitClass')['processSentencenum'].describe())

🔄 Validation JSON 파싱 중...


VL 파싱: 100%|██████████| 36910/36910 [00:04<00:00, 7682.76it/s]



✅ VL 파싱 완료!
  성공: 36,910건 / 실패: 0건

[VL 라벨 분포]
clickbaitClass
0    18741
1    18169

[VL processSentencenum] 낚시성 vs 정상
                  count     mean       std  min  25%  50%  75%   max
clickbaitClass                                                      
0               18741.0  2.41033  1.402897  1.0  1.0  2.0  3.0  13.0
1               18169.0  0.00000  0.000000  0.0  0.0  0.0  0.0   0.0


## 7. 피처 분포 확인 (EDA)

In [7]:
print('=== insertPositionRatio 분포 (낚시성 vs 정상) ===')
print(tr_df.groupby('clickbaitClass')['insertPositionRatio'].describe())

print('\n=== insertedSent 빈 문자열 비율 ===')
print(f"낚시성(0): {(tr_df[tr_df['clickbaitClass']==0]['insertedSent']=='').sum():,}건 빔")
print(f"정상(1):   {(tr_df[tr_df['clickbaitClass']==1]['insertedSent']=='').sum():,}건 빔")

# 샘플 확인
print('\n=== 낚시성 기사 샘플 (processSentencenum 확인) ===')
sample = tr_df[tr_df['clickbaitClass']==0].iloc[0]
print(f'newsID: {sample["newsID"]}')
print(f'processSentencenum: {sample["processSentencenum"]}')
print(f'sentenceCount: {sample["sentenceCount"]}')
print(f'insertPositionRatio: {sample["insertPositionRatio"]:.3f}')
print(f'prevSent: {sample["prevSent"][:50]}...')
print(f'insertedSent: {sample["insertedSent"][:50]}...')
print(f'nextSent: {sample["nextSent"][:50]}...')

=== insertPositionRatio 분포 (낚시성 vs 정상) ===
                   count      mean       std       min    25%     50%  \
clickbaitClass                                                          
0               149928.0  0.208008  0.109032  0.020833  0.125  0.1875   
1               145347.0  0.000000  0.000000  0.000000  0.000  0.0000   

                     75%       max  
clickbaitClass                      
0               0.266667  0.666667  
1               0.000000  0.000000  

=== insertedSent 빈 문자열 비율 ===
낚시성(0): 0건 빔
정상(1):   145,347건 빔

=== 낚시성 기사 샘플 (processSentencenum 확인) ===
newsID: SO_M07_610478
processSentencenum: 3
sentenceCount: 11
insertPositionRatio: 0.273
prevSent: 일부 산업 분야의 경우 탈규제가 주요 이슈로 대두되고 있지만, 국민 건강과 직결되는 식품의...
insertedSent: 그런데 최근 식품산업의 성장과 함께 관련 규제 법령 역시 발 빠르게 변화하고 있다....
nextSent: 식품산업에서 영위하는 기업 또는 개인들의 경우, 이러한 규제의 변화를 좇아가야만 하는 상황...


## 8. CSV 저장

In [8]:
TR_SAVE_PATH = '/content/drive/MyDrive/2026 4-1 캡스톤/DATA/TR_p2_original.csv'
VL_SAVE_PATH = '/content/drive/MyDrive/2026 4-1 캡스톤/DATA/VL_p2_original.csv'

tr_df.to_csv(TR_SAVE_PATH, index=False, encoding='utf-8-sig')
vl_df.to_csv(VL_SAVE_PATH, index=False, encoding='utf-8-sig')

print(f'✅ 저장 완료!')
print(f'  TR → {TR_SAVE_PATH}  ({len(tr_df):,}건,  {os.path.getsize(TR_SAVE_PATH)/1024/1024:.1f} MB)')
print(f'  VL → {VL_SAVE_PATH}  ({len(vl_df):,}건,  {os.path.getsize(VL_SAVE_PATH)/1024/1024:.1f} MB)')
print(f'\n컬럼 목록: {list(tr_df.columns)}')

✅ 저장 완료!
  TR → /content/drive/MyDrive/2026 4-1 캡스톤/DATA/TR_p2_original.csv  (295,275건,  800.7 MB)
  VL → /content/drive/MyDrive/2026 4-1 캡스톤/DATA/VL_p2_original.csv  (36,910건,  100.2 MB)

컬럼 목록: ['newsID', 'newsCategory', 'newsSubcategory', 'newsTitle', 'newsContent', 'partNum', 'processType', 'processLevel', 'sentenceCount', 'clickbaitClass', 'processSentencenum', 'insertPositionRatio', 'insertedSent', 'prevSent', 'nextSent']
